# 01 · Dataset acquisition, preprocessing and audit (Phase 3–4)
Real data only: MIT Supercloud labelled subset, streamed from the public S3 bucket.
Each raw CSV is downloaded inside Colab, hashed (SHA-256), cleaned onto a 1 s grid, written as Parquet to Drive,
and an audit record is appended to `work/reports/preprocess_audit.jsonl`. Re-running resumes where it stopped.

In [ ]:
# --- Setup (run in Google Colab, NOT on a laptop) ---
# 1. Upload the project's `research/` folder to Google Drive at MyDrive/pred_cooling/research
# 2. Runtime > Change runtime type > GPU (needed only for notebook 03)
from google.colab import drive
drive.mount('/content/drive')
import sys, subprocess
ROOT = '/content/drive/MyDrive/pred_cooling'
sys.path.insert(0, f'{ROOT}/research/src')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pyarrow', 'onnx', 'onnxruntime'], check=True)
from pcool.config import Config
# work_dir holds cleaned data, features, models and reports. It stays in YOUR Drive
# (dataset licence CC BY-NC-ND 4.0: do not share the cleaned data publicly).
cfg = Config(work_dir=f'{ROOT}/work')
print('config hash', cfg.hash())

In [ ]:
from pcool import pipeline
summary = pipeline.stage_preprocess(cfg, workers=16)   # resumable; re-run after a Colab disconnect
summary

## Audit numbers (measured)
These fill the **[Colab]** cells of `research/dataset_audit/dataset_audit.md`.

In [ ]:
import json, pandas as pd
from pcool import preprocess
audit = preprocess.load_audit(cfg)
print('files with errors:', audit['error'].notna().sum() if 'error' in audit else 0)
print(json.dumps(summary, indent=2, default=str))